In [ ]:
# ===========================================
# Notebook Name:
# 02_archetype_analysis
#
# Purpose:
# Metagame-share analysis by archetype, using
# gold.v_deck_archetypes_named (the latest
# clustering run joined to human-reviewed
# archetype identities) rather than the raw,
# per-run cluster_id, which is unstable across
# reruns.
#
# This notebook is exploratory only and is NOT
# part of the Daily/Weekly Workflow.
#
# Input:
# pokemon.gold.v_deck_archetypes_named
# pokemon.gold.deck_pokemon_features
# pokemon.silver.tournament_results
# pokemon.silver.tournaments
# ===========================================

In [ ]:
from pyspark.sql import functions as F

DECK_ARCHETYPES_NAMED_VIEW = "pokemon.gold.v_deck_archetypes_named"
DECK_POKEMON_FEATURES_TABLE = "pokemon.gold.deck_pokemon_features"
TOURNAMENT_RESULTS_TABLE = "pokemon.silver.tournament_results"
TOURNAMENTS_TABLE = "pokemon.silver.tournaments"

TOP_POKEMON_PER_ARCHETYPE = 5

deck_archetypes_named_df = spark.table(DECK_ARCHETYPES_NAMED_VIEW)
deck_pokemon_features_df = spark.table(DECK_POKEMON_FEATURES_TABLE)
tournament_results_df = spark.table(TOURNAMENT_RESULTS_TABLE)
tournaments_df = spark.table(TOURNAMENTS_TABLE)

In [ ]:
latest_model_run_row = (
    deck_archetypes_named_df
    .select("model_run_id", "clustered_at")
    .distinct()
    .orderBy(F.col("clustered_at").desc())
    .first()
)

if latest_model_run_row is None:
    raise ValueError(
        f"{DECK_ARCHETYPES_NAMED_VIEW} is empty. "
        "Run 04_ml/00_build_deck_archetypes first."
    )

latest_model_run_id = latest_model_run_row["model_run_id"]

latest_archetypes_df = (
    deck_archetypes_named_df
    .filter(F.col("model_run_id") == latest_model_run_id)
    .withColumn(
        "archetype_label",
        F.coalesce(
            F.col("archetype_name"),
            F.lit("(pending review)"),
        ),
    )
)

print("Analyzing model_run_id:", latest_model_run_id)

In [ ]:
archetype_deck_counts_df = (
    latest_archetypes_df
    .groupBy("archetype_label", "mapping_status")
    .agg(F.countDistinct("deck_hash").alias("deck_count"))
    .orderBy(F.col("deck_count").desc())
)

display(archetype_deck_counts_df)

In [ ]:
from pyspark.sql.window import Window

representative_pokemon_window_df = (
    deck_pokemon_features_df
    .filter(F.col("is_present") == 1)
    .join(
        latest_archetypes_df.select("deck_hash", "archetype_label"),
        on="deck_hash",
        how="inner",
    )
    .groupBy("archetype_label", "card_name")
    .agg(F.countDistinct("deck_hash").alias("decks_using_pokemon"))
)

representative_pokemon_rank_window = (
    Window
    .partitionBy("archetype_label")
    .orderBy(F.col("decks_using_pokemon").desc())
)

top_pokemon_per_archetype_df = (
    representative_pokemon_window_df
    .withColumn(
        "rank_in_archetype",
        F.row_number().over(representative_pokemon_rank_window),
    )
    .filter(F.col("rank_in_archetype") <= TOP_POKEMON_PER_ARCHETYPE)
    .groupBy("archetype_label")
    .agg(
        F.sort_array(
            F.collect_list("card_name")
        ).alias("top_pokemon")
    )
)

display(
    archetype_deck_counts_df
    .join(top_pokemon_per_archetype_df, on="archetype_label", how="left")
    .orderBy(F.col("deck_count").desc())
)

In [ ]:
weekly_archetype_share_df = (
    tournament_results_df
    .select("tournament_id", "deck_hash")
    .join(
        tournaments_df.select(
            "tournament_id",
            "event_date",
            "event_date_week",
        ),
        on="tournament_id",
        how="inner",
    )
    .join(
        latest_archetypes_df.select("deck_hash", "archetype_label"),
        on="deck_hash",
        how="inner",
    )
    .groupBy("event_date_week", "archetype_label")
    .agg(F.count("*").alias("result_count"))
)

weekly_total_df = (
    weekly_archetype_share_df
    .groupBy("event_date_week")
    .agg(F.sum("result_count").alias("weekly_total_results"))
)

weekly_archetype_share_pct_df = (
    weekly_archetype_share_df
    .join(weekly_total_df, on="event_date_week", how="left")
    .withColumn(
        "share_pct",
        F.round(
            F.col("result_count") * 100.0 / F.col("weekly_total_results"),
            1,
        ),
    )
    .orderBy("event_date_week", F.col("share_pct").desc())
)

display(weekly_archetype_share_pct_df)

In [ ]:
display(
    weekly_archetype_share_pct_df
    .groupBy("event_date_week")
    .pivot("archetype_label")
    .agg(F.first("share_pct"))
    .orderBy("event_date_week")
)